In [12]:
import pandas as pd
import numpy as np
import ast

# 1. Load the raw files
df_games = pd.read_csv('games.csv')
df_vgsales = pd.read_csv('vgsales.csv')

print(f"Initial games shape: {df_games.shape}")
print(f"Initial vgsales shape: {df_vgsales.shape}")

# Inspect column types and initial missing values
print("\n--- Missing values in games.csv ---")
print(df_games.isnull().sum()[df_games.isnull().sum() > 0])

print("\n--- Missing values in vgsales.csv ---")
print(df_vgsales.isnull().sum()[df_vgsales.isnull().sum() > 0])

Initial games shape: (1512, 14)
Initial vgsales shape: (16598, 11)

--- Missing values in games.csv ---
Team        1
Rating     13
Summary     1
dtype: int64

--- Missing values in vgsales.csv ---
Year         271
Publisher     58
dtype: int64


In [19]:
import pandas as pd
import numpy as np
import ast

# 1. Load raw data
df_games = pd.read_csv('games.csv')
df_vgsales = pd.read_csv('vgsales.csv')

# ==========================================
# A. CLEAN games.csv
# ==========================================
# Drop unnecessary index column and text columns causing Workbench parser crash
cols_to_drop = [c for c in ['Unnamed: 0', 'Summary', 'Reviews'] if c in df_games.columns]
df_games = df_games.drop(columns=cols_to_drop)

# Convert 'K' string representations into integers
def parse_metric(val):
    if pd.isna(val):
        return 0
    val_str = str(val).strip().upper()
    if 'K' in val_str:
        return int(float(val_str.replace('K', '')) * 1000)
    try:
        return int(float(val_str))
    except ValueError:
        return 0

metric_cols = ['Plays', 'Playing', 'Backlogs', 'Wishlist', 'Times Listed', 'Number of Reviews']
for col in metric_cols:
    if col in df_games.columns:
        df_games[col] = df_games[col].apply(parse_metric)

# Clean bracketed string lists (Genres & Team)
def clean_list_format(val):
    if pd.isna(val):
        return 'Unknown'
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, list):
            return ', '.join([str(item).strip() for item in parsed if item])
    except Exception:
        pass
    return str(val).strip("[]'\" ")

df_games['Team'] = df_games['Team'].apply(clean_list_format).replace('', 'Unknown')
df_games['Genres'] = df_games['Genres'].apply(clean_list_format).replace('', 'Unknown')
df_games['Primary_Genre'] = df_games['Genres'].apply(lambda x: x.split(',')[0].strip() if x else 'Unknown')

# Handle numeric ratings & dates
df_games['Rating'] = df_games['Rating'].fillna(df_games['Rating'].median()).round(2)
df_games['Release_Date'] = pd.to_datetime(df_games['Release Date'], errors='coerce').dt.strftime('%Y-%m-%d')
if 'Release Date' in df_games.columns:
    df_games = df_games.drop(columns=['Release Date'])

# Normalize Title and drop duplicates
df_games['Title'] = df_games['Title'].astype(str).str.strip().str.replace('"', '', regex=False)
df_games = df_games.drop_duplicates(subset=['Title'], keep='first')

# Rename to clean SQL column names
df_games = df_games.rename(columns={
    'Times Listed': 'Times_Listed',
    'Number of Reviews': 'Number_of_Reviews'
})


# ==========================================
# B. CLEAN vgsales.csv
# ==========================================
df_vgsales['Name'] = df_vgsales['Name'].astype(str).str.strip().str.replace('"', '', regex=False)
df_vgsales['Platform'] = df_vgsales['Platform'].astype(str).str.strip().str.upper()
df_vgsales['Genre'] = df_vgsales['Genre'].astype(str).str.strip()
df_vgsales['Publisher'] = df_vgsales['Publisher'].fillna('Unknown').astype(str).str.strip()

median_year = int(df_vgsales['Year'].median())
df_vgsales['Year'] = df_vgsales['Year'].fillna(median_year).astype(int)

sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
for col in sales_cols:
    df_vgsales[col] = pd.to_numeric(df_vgsales[col], errors='coerce').fillna(0.0).round(2)

df_vgsales = df_vgsales.drop_duplicates(subset=['Name', 'Platform'], keep='first')


# ==========================================
# C. VERIFICATION & EXPORT
# ==========================================
print("Games Nulls:\n", df_games.isnull().sum()[df_games.isnull().sum() > 0])
print("Sales Nulls:\n", df_vgsales.isnull().sum()[df_vgsales.isnull().sum() > 0])

# Export clean CSVs (standard utf-8)
df_games.to_csv('clean_games.csv', index=False, encoding='utf-8')
df_vgsales.to_csv('clean_vgsales.csv', index=False, encoding='utf-8')

# Merged CSV
df_merged = pd.merge(df_vgsales, df_games, left_on='Name', right_on='Title', how='inner')
df_merged.to_csv('clean_merged_vgsales.csv', index=False, encoding='utf-8')

print("\nDone! All files re-saved cleanly.")

Games Nulls:
 Release_Date    3
dtype: int64
Sales Nulls:
 Series([], dtype: int64)

Done! All files re-saved cleanly.
